# Seasonal Decay Comparison (Rain)

This notebook compares **winter decay**, **standard decay**, and **summer decay** settings for `COV19` in rain conditions.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sys.path.append("..")
from plotting_tools import *


In [ ]:
experiment_series = "pop8"
result_path = f"../../../preprocessing/preprocessed_data/{experiment_series}/substances"

files = {
    "Winter": "winter_decay_Rain_output_scaled.csv",
    "Standard": "decay_Rain_output_scaled.csv",
    "Summer": "summer_decay_Rain_output_scaled.csv",
}

locations_to_plot = ["N_D"]
variable = "COV19"

plot_out_dir = "../../plots/Supplement"
os.makedirs(plot_out_dir, exist_ok=True)


In [ ]:
def load_filtered_timeseries(filename, locations=("N_D", "N_Ua"), variable_name="COV19", chunksize=2_000_000):
    """Load only selected variable and locations to keep memory usage manageable."""
    filepath = f"{result_path}/{filename}"

    manholes = [m for m, loc in manhole_names.items() if loc in locations]
    usecols = ["time_in_minutes", "variable", "value", "manhole", "simulation_id"]

    frames = []
    for chunk in pd.read_csv(filepath, usecols=usecols, chunksize=chunksize):
        mask = (chunk["variable"] == variable_name) & (chunk["manhole"].isin(manholes))
        if mask.any():
            c = chunk.loc[mask].copy()
            c["location"] = c["manhole"].map(manhole_names)
            frames.append(c)

    if not frames:
        return pd.DataFrame(columns=usecols + ["location", "Date"])

    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(start_date) + pd.to_timedelta(df["time_in_minutes"], unit="min")
    return df


In [ ]:
data = {
    label: load_filtered_timeseries(fname, locations=locations_to_plot, variable_name=variable)
    for label, fname in files.items()
}

for label, df in data.items():
    print(f"{label}: {len(df):,} rows")


In [ ]:
# Relative differences versus the standard setting
join_keys = ["time_in_minutes", "manhole", "simulation_id", "location", "Date"]

winter_vs_standard = data["Winter"].merge(
    data["Standard"],
    on=join_keys,
    suffixes=("_winter", "_standard"),
)

summer_vs_standard = data["Summer"].merge(
    data["Standard"],
    on=join_keys,
    suffixes=("_summer", "_standard"),
)

winter_vs_standard["relative_difference"] = np.where(
    winter_vs_standard["value_standard"] != 0,
    (winter_vs_standard["value_winter"] - winter_vs_standard["value_standard"]) / winter_vs_standard["value_standard"],
    np.nan,
)

summer_vs_standard["relative_difference"] = np.where(
    summer_vs_standard["value_standard"] != 0,
    (summer_vs_standard["value_summer"] - summer_vs_standard["value_standard"]) / summer_vs_standard["value_standard"],
    np.nan,
)

winter_vs_standard["comparison"] = "Winter vs Standard"
summer_vs_standard["comparison"] = "Summer vs Standard"

relative_df = pd.concat(
    [
        winter_vs_standard[["Date", "location", "relative_difference", "comparison"]],
        summer_vs_standard[["Date", "location", "relative_difference", "comparison"]],
    ],
    ignore_index=True,
)


In [ ]:
relative_df

In [ ]:
# Summary table in percent
summary = (
    relative_df
    .dropna(subset=["relative_difference"])
    .groupby(["location", "comparison"])["relative_difference"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

(summary * 100).round(2)


In [ ]:
# Combined figure with two rows
# Row 1: Winter (left), Standard (middle), Summer (right)
# Row 2: Relative differences vs standard (single axis spanning full width)
from matplotlib.gridspec import GridSpec

order = ["Winter", "Standard", "Summer"]
if len(locations_to_plot) != 1:
    raise ValueError("This combined 2-row layout expects exactly one location in locations_to_plot.")

loc = locations_to_plot[0]

fig = plt.figure(figsize=(16, 7))
gs = GridSpec(2, 3, figure=fig, height_ratios=[2.2, 1.3])

second_of_month = mdates.DayLocator(bymonthday=2)

# Top row: scenario comparison
for col_idx, season in enumerate(order):
    ax = fig.add_subplot(gs[0, col_idx])
    df_plot = data[season].loc[data[season]["location"] == loc]

    sns.lineplot(
        data=df_plot,
        x="Date",
        y="value",
        ax=ax,
        color=blue,
        estimator=np.mean,
        errorbar=("pi", 90),
    )

    ax.set_title(season)
    ax.set_xlabel("")
    if col_idx == 0:
        ax.set_ylabel("Virus levels\n[copies/l]")
    else:
        ax.set_ylabel("")

    ax.xaxis.set_major_locator(second_of_month)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45)

# Bottom row: relative differences (span all columns)
ax_rel = fig.add_subplot(gs[1, :])
palette = {
    "Winter vs Standard": teal,
    "Summer vs Standard": orange,
}

rel_plot = relative_df.loc[relative_df["location"] == loc].dropna(subset=["relative_difference"])

sns.lineplot(
    data=rel_plot,
    x="Date",
    y="relative_difference",
    hue="comparison",
    estimator=np.mean,
    errorbar=("pi", 90),
    palette=palette,
    ax=ax_rel,
)

ax_rel.set_xlabel("")
ax_rel.set_ylabel("Relative difference")
ax_rel.xaxis.set_major_locator(second_of_month)
ax_rel.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
for lbl in ax_rel.get_xticklabels():
    lbl.set_rotation(45)
ax_rel.axhline(0, color=dark_grey, linewidth=0.8, linestyle="--")
ax_rel.legend(title="")

plt.tight_layout()
fig.savefig(f"{plot_out_dir}/{experiment_series}_winter_standard_summer_combined_with_relative_COV19.png", dpi=300)
plt.show()
